# KPSS CANAVARI - Vector Database Yükleme

Bu notebook embedding'leri Vector Database'e (Pinecone veya ChromaDB) yükleyecek.

## Seçenekler:

### 1. Pinecone (Önerilir - Production için)
- ✅ Çok hızlı arama
- ✅ Managed service (bakım yok)
- ✅ Scalable
- ⚠️ $70/ay (Free tier: 1M vector'e kadar)

### 2. ChromaDB (Ücretsiz - Development için)
- ✅ Tamamen ücretsiz
- ✅ Self-hosted
- ⚠️ Biraz daha yavaş
- ⚠️ Manuel bakım gerekir

Bu notebook her iki seçeneği de destekler.

In [ ]:
# KONFIGURASYON
# Hangi Vector DB kullanacaksınız?

USE_PINECONE = True  # False yaparsanız ChromaDB kullanılır

# Pinecone API Key (sadece Pinecone kullanıyorsanız)
PINECONE_API_KEY = "your-pinecone-api-key-here"  # https://www.pinecone.io/ adresinden alın
PINECONE_ENVIRONMENT = "us-east-1"  # Bölgenizi seçin
PINECONE_INDEX_NAME = "kpss-canavari"

print(f'🗄️ Vector DB: {"Pinecone" if USE_PINECONE else "ChromaDB"}')

In [ ]:
# 1. KÜTÜPHANELERI YÜKLE
if USE_PINECONE:
    !pip install -q pinecone-client tqdm
else:
    !pip install -q chromadb tqdm

print('✅ Kütüphaneler yüklendi!')

In [ ]:
# 2. GOOGLE DRIVE BAĞLA VE VERİLERİ YÜKLE
from google.colab import drive
import json
import numpy as np
import os

drive.mount('/content/drive')

OUTPUT_PATH = '/content/drive/MyDrive/KPSS_Processed'

# Chunk'ları yükle
print('📥 Chunk\'lar yükleniyor...')
with open(os.path.join(OUTPUT_PATH, 'chunks.json'), 'r', encoding='utf-8') as f:
    chunks = json.load(f)

# Embedding'leri yükle
print('📥 Embedding\'ler yükleniyor...')
embeddings = np.load(os.path.join(OUTPUT_PATH, 'embeddings.npy'))

# Metadata yükle
with open(os.path.join(OUTPUT_PATH, 'embeddings_metadata.json'), 'r') as f:
    metadata = json.load(f)

print(f'\n✅ Veriler yüklendi!')
print(f'📦 Chunk sayısı: {len(chunks):,}')
print(f'📏 Embedding boyutu: {embeddings.shape}')
print(f'🔢 Embedding dimension: {metadata["embedding_dimension"]}')

## PINECONE İLE YÜKLEME

In [ ]:
# 3A. PINECONE KURULUM (Sadece USE_PINECONE = True ise çalıştırın)
if USE_PINECONE:
    import pinecone
    from tqdm import tqdm
    
    # Pinecone'a bağlan
    print('🔌 Pinecone\'a bağlanıyor...')
    pinecone.init(
        api_key=PINECONE_API_KEY,
        environment=PINECONE_ENVIRONMENT
    )
    
    # Index oluştur (eğer yoksa)
    if PINECONE_INDEX_NAME not in pinecone.list_indexes():
        print(f'📦 Index oluşturuluyor: {PINECONE_INDEX_NAME}')
        pinecone.create_index(
            name=PINECONE_INDEX_NAME,
            dimension=metadata['embedding_dimension'],
            metric='cosine'
        )
    else:
        print(f'✅ Index mevcut: {PINECONE_INDEX_NAME}')
    
    # Index'e bağlan
    index = pinecone.Index(PINECONE_INDEX_NAME)
    
    print('✅ Pinecone hazır!')
    print(f'📊 Index stats: {index.describe_index_stats()}')

In [ ]:
# 4A. PINECONE'A YÜKLE
if USE_PINECONE:
    import time
    
    BATCH_SIZE = 100  # Pinecone batch size
    
    print(f'🚀 Pinecone\'a yükleme başlıyor...')
    print(f'📦 Batch size: {BATCH_SIZE}')
    
    start_time = time.time()
    
    # Batch processing
    for i in tqdm(range(0, len(chunks), BATCH_SIZE), desc='Uploading'):
        batch_end = min(i + BATCH_SIZE, len(chunks))
        
        # Batch hazırla
        batch_data = []
        for j in range(i, batch_end):
            batch_data.append((
                chunks[j]['chunk_id'],  # ID
                embeddings[j].tolist(),  # Vector
                {  # Metadata
                    'text': chunks[j]['text'],
                    'source_file': chunks[j]['source_file'],
                    'chunk_index': chunks[j]['chunk_index']
                }
            ))
        
        # Pinecone'a yükle
        try:
            index.upsert(vectors=batch_data)
        except Exception as e:
            print(f'❌ Hata (batch {i}-{batch_end}): {str(e)}')
            break
    
    elapsed = time.time() - start_time
    
    print(f'\n✅ Yükleme tamamlandı!')
    print(f'⏱️ Toplam süre: {elapsed/60:.1f} dakika')
    print(f'📊 Index stats: {index.describe_index_stats()}')

## CHROMADB İLE YÜKLEME

In [ ]:
# 3B. CHROMADB KURULUM (Sadece USE_PINECONE = False ise çalıştırın)
if not USE_PINECONE:
    import chromadb
    from chromadb.config import Settings
    
    # ChromaDB klasörü
    CHROMA_PATH = os.path.join(OUTPUT_PATH, 'chromadb')
    os.makedirs(CHROMA_PATH, exist_ok=True)
    
    # ChromaDB client oluştur
    print(f'🔌 ChromaDB başlatılıyor...')
    client = chromadb.PersistentClient(path=CHROMA_PATH)
    
    # Collection oluştur (eğer yoksa)
    collection_name = "kpss_canavari"
    
    try:
        collection = client.get_collection(name=collection_name)
        print(f'✅ Collection mevcut: {collection_name}')
    except:
        collection = client.create_collection(
            name=collection_name,
            metadata={"description": "KPSS Canavari knowledge base"}
        )
        print(f'📦 Collection oluşturuldu: {collection_name}')
    
    print('✅ ChromaDB hazır!')

In [ ]:
# 4B. CHROMADB'YE YÜKLE
if not USE_PINECONE:
    import time
    from tqdm import tqdm
    
    BATCH_SIZE = 500  # ChromaDB daha büyük batch'leri destekler
    
    print(f'🚀 ChromaDB\'ye yükleme başlıyor...')
    print(f'📦 Batch size: {BATCH_SIZE}')
    
    start_time = time.time()
    
    # Batch processing
    for i in tqdm(range(0, len(chunks), BATCH_SIZE), desc='Uploading'):
        batch_end = min(i + BATCH_SIZE, len(chunks))
        
        # Batch verilerini hazırla
        batch_ids = [chunks[j]['chunk_id'] for j in range(i, batch_end)]
        batch_embeddings = [embeddings[j].tolist() for j in range(i, batch_end)]
        batch_documents = [chunks[j]['text'] for j in range(i, batch_end)]
        batch_metadatas = [
            {
                'source_file': chunks[j]['source_file'],
                'chunk_index': chunks[j]['chunk_index']
            }
            for j in range(i, batch_end)
        ]
        
        # ChromaDB'ye yükle
        try:
            collection.add(
                ids=batch_ids,
                embeddings=batch_embeddings,
                documents=batch_documents,
                metadatas=batch_metadatas
            )
        except Exception as e:
            print(f'❌ Hata (batch {i}-{batch_end}): {str(e)}')
            break
    
    elapsed = time.time() - start_time
    
    print(f'\n✅ Yükleme tamamlandı!')
    print(f'⏱️ Toplam süre: {elapsed/60:.1f} dakika')
    print(f'📊 Collection count: {collection.count()}')

In [ ]:
# 5. TEST: BENZERLİK ARAMASI
from sentence_transformers import SentenceTransformer

# Embedding modelini yükle (test için)
print('📦 Model yükleniyor (test için)...')
model = SentenceTransformer(metadata['model_name'])

# Test sorguları
test_queries = [
    "Türkiye Cumhuriyeti'nin başkenti neresidir?",
    "Atatürk hangi tarihte doğdu?",
    "KPSS sınavı kaç bölümden oluşur?"
]

for query in test_queries:
    print(f'\n🔍 Sorgu: "{query}"')
    
    # Sorgu embedding'i
    query_embedding = model.encode([query])[0]
    
    if USE_PINECONE:
        # Pinecone'da ara
        results = index.query(
            vector=query_embedding.tolist(),
            top_k=3,
            include_metadata=True
        )
        
        print('\n📊 En benzer 3 sonuç:')
        for i, match in enumerate(results['matches']):
            print(f'\n{i+1}. Skor: {match["score"]:.4f}')
            print(f'   Kaynak: {match["metadata"]["source_file"]}')
            print(f'   Metin: {match["metadata"]["text"][:150]}...')
    
    else:
        # ChromaDB'de ara
        results = collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=3
        )
        
        print('\n📊 En benzer 3 sonuç:')
        for i in range(len(results['ids'][0])):
            print(f'\n{i+1}. Distance: {results["distances"][0][i]:.4f}')
            print(f'   Kaynak: {results["metadatas"][0][i]["source_file"]}')
            print(f'   Metin: {results["documents"][0][i][:150]}...')

print('\n✅ Test başarılı!')

In [ ]:
# 6. KONFIGURASYON KAYDET
config = {
    'vector_db': 'Pinecone' if USE_PINECONE else 'ChromaDB',
    'embedding_model': metadata['model_name'],
    'embedding_dimension': metadata['embedding_dimension'],
    'total_vectors': len(chunks)
}

if USE_PINECONE:
    config['pinecone_index'] = PINECONE_INDEX_NAME
    config['pinecone_environment'] = PINECONE_ENVIRONMENT
else:
    config['chromadb_path'] = CHROMA_PATH
    config['collection_name'] = collection_name

config_file = os.path.join(OUTPUT_PATH, 'vector_db_config.json')
with open(config_file, 'w') as f:
    json.dump(config, f, indent=2)

print(f'💾 Konfigürasyon kaydedildi: {config_file}')
print('\n📋 Konfigürasyon:')
for key, value in config.items():
    print(f'  {key}: {value}')

## ✅ TAMAMLANDI!

Vector Database hazır! Sonraki adım: `04_rag_testing.ipynb` ile Gemini API entegrasyonu ve tam RAG pipeline testi.